# Qwen3.5 Audit v11 CVS Context v3.1 Train Full - base v3 generation

This section runs the base `cvsctx_v3` segmentation/extraction pass for the split. The following section applies the v3.1 Hook-to-Maryland override and writes the final synthetic export.

Train uses generated SAGES train transition clip records at `train/audit_v11_clips`. Generate them with `scripts/generate_train_transition_clip_records.py --split train`.


In [ ]:

from openai import OpenAI
import hashlib
import json
import os
import sys
import time
from pathlib import Path
from tqdm.auto import tqdm

try:
    import pandas as pd
except Exception:
    pd = None

ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cvs_act.action_segment_eval import (
    ACTORS,
    build_deterministic_structured_extraction,
    build_extraction_prompt as shared_build_extraction_prompt,
    build_messages_for_record,
    build_simple_options,
    convert_record_to_simple_actions,
    default_segmentation_prompts,
    extract_json_object,
    load_audit_records,
    naturalize_simple_actions,
    is_deterministic_structured_method,
    normalize_extraction,
    read_json,
    segmentation_source_method,
    write_json,
)

with open('/mnt/md0/weiqiuy/ips/carnaroli.txt') as input_file:
    carnaroli_ip = input_file.read().strip()
client = OpenAI(base_url=f'http://{carnaroli_ip}:8001/v1', api_key='brachiokey')

MODEL_ID = 'Qwen/Qwen3.6-35B-A3B-FP8'
REQUEST_TEMPERATURE = 0
REQUEST_ENABLE_THINKING = False
ANNOTATION_ROOT = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1'
AUDIT_V11_DIR = ANNOTATION_ROOT / 'train/audit_v11_clips'
FRAMES_DIR = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/frames/train'
SPEC_PATH = ANNOTATION_ROOT / 'specs/eval_spec.md'
ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_train_full'
SEGMENT_CACHE_PATH = ARTIFACT_DIR / 'segment_cache.json'
EXTRACTION_CACHE_PATH = ARTIFACT_DIR / 'extraction_cache.json'
RESULTS_PATH = ARTIFACT_DIR / 'results.json'

FORCE_SEGMENT = False
FORCE_EXTRACTION = False
FORCE_JUDGE = False
DEBUG_RUN = False #True
DEBUG_RECORD_LIMIT = 5
MAX_RECORDS = None
WRITE_DERIVED_ARTIFACTS = True
BASE_METHODS = ['structured_prediction']
LABELING_VERSION = 'code_labels_v3_cvs_perframe_scores_v3_right_rules_train_full'
METHODS = [f'{method}_cvs_context' for method in BASE_METHODS]

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
print('model:', MODEL_ID)
print('spec:', SPEC_PATH)
print('annotations:', AUDIT_V11_DIR)
print('frames:', FRAMES_DIR)
print('artifacts:', ARTIFACT_DIR)


In [2]:

records = load_audit_records(AUDIT_V11_DIR)
if DEBUG_RUN:
    records = records[:DEBUG_RECORD_LIMIT]
elif MAX_RECORDS:
    records = records[:MAX_RECORDS]

gt_simple_records = [convert_record_to_simple_actions(record) for record in records]
natural_gt_simple_records = [naturalize_simple_actions(record) for record in gt_simple_records]
HF_TAXONOMY_PATH = ROOT_DIR / 'hf_repos/cvs-act/taxonomy/action_taxonomy.json'


def build_hf_taxonomy_simple_options(taxonomy_path):
    taxonomy = read_json(taxonomy_path)
    actors = taxonomy['actors']
    left_codes = list(actors['left']['components']['retraction_direction_code'])
    right_tools = list(actors['right']['components']['tool_type'])
    right_actions = list(actors['right']['components']['action_code'])
    right_targets = list(actors['right']['components']['target_structure'])
    right_contexts = list(actors['right']['components']['target_context'])
    camera_actions = list(actors['camera']['components']['action_code'])
    other_actions = list((actors.get('other', {}).get('components', {}) or {}).get('action_code', []))
    if 'ICG_SWITCH' not in other_actions:
        other_actions.append('ICG_SWITCH')
    right_triplets = [
        [tool_type, action_code, target_structure]
        for tool_type in right_tools
        for action_code in right_actions
        for target_structure in right_targets
    ]
    return {
        'source': f'HF taxonomy gold options: {taxonomy_path}',
        'taxonomy_version': taxonomy.get('taxonomy_version'),
        'interface_options': {
            'retraction_direction_options': left_codes,
            'camera_action_codes': camera_actions,
            'tool_type': right_tools,
            'action_code': right_actions,
            'target_structure': right_targets,
            'target_context': right_contexts,
        },
        'left_retraction_direction_code_options': left_codes,
        'right_triplet_options': right_triplets,
        'camera_action_code_options': camera_actions,
        'other_action_code_options': other_actions,
    }


simple_options = build_hf_taxonomy_simple_options(HF_TAXONOMY_PATH)
if WRITE_DERIVED_ARTIFACTS or not (ARTIFACT_DIR / 'audit_v11_simple_actions.json').exists():
    write_json(ARTIFACT_DIR / 'audit_v11_simple_actions.json', gt_simple_records)
if WRITE_DERIVED_ARTIFACTS or not (ARTIFACT_DIR / 'audit_v11_simple_actions_natural_language.json').exists():
    write_json(ARTIFACT_DIR / 'audit_v11_simple_actions_natural_language.json', natural_gt_simple_records)
if WRITE_DERIVED_ARTIFACTS or not (ARTIFACT_DIR / 'audit_v11_simple_action_options.json').exists():
    write_json(ARTIFACT_DIR / 'audit_v11_simple_action_options.json', simple_options)


base_prompts = default_segmentation_prompts(simple_options)

CVS_CRITERION_DESCRIPTIONS = {
    'C1': 'Two and only two tubular structures are visible entering the gallbladder.',
    'C2': 'The hepatocystic triangle is cleared of fat and fibrous tissue.',
    'C3': 'The lower third of the gallbladder is detached from the liver bed.',
}

CVS_LABELS_DIR = Path('/mnt/md0/weiqiuy/datasets/CVS_Challenge_SAGES_v1/train/labels')
_cvs_score_cache: dict = {}


def load_cvs_frame_scores(video_id):
    """Return {frame_id: {'c1': k, 'c2': k, 'c3': k}} where k in {0,1,2,3} = #raters satisfied."""
    if video_id in _cvs_score_cache:
        return _cvs_score_cache[video_id]
    import csv
    path = CVS_LABELS_DIR / video_id / 'frame.csv'
    scores: dict = {}
    if path.exists():
        with open(path) as fh:
            for row in csv.DictReader(fh):
                fid = int(row['frame_id'])
                scores[fid] = {
                    'c1': sum(int(row[f'c1_rater{i}']) for i in (1, 2, 3)),
                    'c2': sum(int(row[f'c2_rater{i}']) for i in (1, 2, 3)),
                    'c3': sum(int(row[f'c3_rater{i}']) for i in (1, 2, 3)),
                }
    _cvs_score_cache[video_id] = scores
    return scores


def build_frame_labels_for_record(record):
    """Return {frame_id: label_str}; anchors get CVS scores, in-between frames omitted (default label used)."""
    scores = load_cvs_frame_scores(record['video_id'])
    labels = {}
    for fid, s in scores.items():
        labels[fid] = f"Frame {fid:06d}, C1={s['c1']}/3, C2={s['c2']}/3, C3={s['c3']}/3"
    return labels


def build_cvs_context_prompt(record, base_prompt):
    criterion = record.get('criterion')
    criterion_lines = '\n'.join(f'- {code}: {desc}' for code, desc in CVS_CRITERION_DESCRIPTIONS.items())
    coarse = record.get('coarse', {})
    context = f'''
CVS criteria definitions:
{criterion_lines}

Per-frame CVS satisfaction labels:
Anchor frames (every 5 seconds, i.e. at the stride-150 frame numbers) are annotated by 3 reviewers and shown with labels like "Frame 002100, C1=1/3, C2=2/3, C3=0/3". The value k/3 is the number of reviewers (of 3) who say that criterion is satisfied at that frame: 3/3 means all three reviewers agree it is satisfied, 0/3 means none, 1/3 means exactly one of three. The interleaved finer-grained frames (between anchors) carry only their frame number with no score because they were not separately rated.

This clip primarily concerns criterion {criterion}. Clip frame range: {coarse.get('start_frame')} to {coarse.get('end_frame')}.

Use the CVS context only as clip-level context. Still base segment boundaries and action timing on the provided frames and visible frame labels.
'''.strip()
    return context + '\n\n' + base_prompt


def prompt_for_record(record, method_name):
    base_method = segmentation_source_method(method_name).removesuffix('_cvs_context')
    return build_cvs_context_prompt(record, base_prompts[base_method])


def build_extraction_prompt(example_id, frame_range, method_name, prediction_text, options):
    prompt = shared_build_extraction_prompt(example_id, frame_range, method_name, prediction_text, options)
    left_rule = '''
Additional left-only rule for this v3 run:
- Use a left change code such as RETRACT_LATERAL_TO_MEDIAL or RETRACT_MEDIAL_TO_LATERAL only for the short segment where the retraction direction visibly transitions from one direction to another.
- Do not use a change code for a long interval that is merely described as holding the gallbladder upward and laterally, medially and upward, or otherwise stably exposed in multiple directions.
- If the segment is mostly stable rather than an actual transition, prefer the matching KEEP_RETRACT_* or RETRACT_* endpoint direction code instead of a *_TO_* code.
'''.strip()
    right_rule = '''\nAdditional right-tool rules for this v3 run:
- First decide whether the right-hand tool is doing a meaningful action. If a tool is merely present without a meaningful interaction, do not force an action label.
- For right-hand tool actions, choose the most appropriate action, target, and target context using the following rules.

1. Dissection: choose the main target first.
- For dissection actions, the tool is usually operating in the hepatocystic triangle region.
- More specifically, it may be operating around the cystic artery, cystic duct, or cystic plate.
- If the hepatocystic triangle is not cleared at all and the tool is dissecting only in the general hepatocystic triangle region, report the target as HepatocysticTriangle. In that case, you do not need a more specific target.
- Even if the triangle is not completely cleared, if you can tell that the tool is operating mostly near the presumed cystic duct, presumed cystic artery, or cystic plate, then prefer the more specific target as the main target.

2. Dissection: add target context when the operating side is visible.
- If the tool is operating mostly on one side of the target, specify that side in the target context.
- For example, the hook may be dissecting near the cystic artery:
  - between presumed cystic duct and presumed cystic artery, or
  - between cystic artery and cystic plate.
- If the tool is operating on both sides, put one side in target_context_1 and the other in target_context_2. The order does not matter.

3. Use hepatocystic triangle with precise context when appropriate.
- If the two tubular structures are clear, but the tool is operating in the general region between them rather than trying to skeletonize either the cystic duct or the cystic artery, choose HepatocysticTriangle as the target and use between presumed cystic duct and presumed cystic artery as the context.
- If the tool is pushing down (retracting) or dissecting near the base of the hepatocystic triangle, that should also be included in the context when visible.

4. Countertraction assist.
- Sometimes an irrigator or a hook is helping the grasper that is retracting the gallbladder neck change direction.
- In that situation, you can label the action as COUNTERTRACTION_ASSIST.

5. Irrigator aspirating.
- If an irrigator is present and blood disappears from the scene, label the action as IRRIGATOR_ASPIRATE.

6. Tool withdrawn and view unblocked.
- If a tool is present at first, does not perform any meaningful action, and is then simply removed from the scene, you can label the action as TOOL_WITHDRAW_UNBLOCKS_VIEW.
- If the tool was blocking only part of the scene, then the target should be only the specific region that was blocked and then unblocked.
- For example, if it was blocking only the cystic plate, while the rest of the field was already visible, then the target should be CysticPlate, not the whole hepatocystic triangle.

7. Coagulating to stop bleeding.
- Sometimes an electrocautery tool may be used to coagulate blood and stop bleeding.
- In that case, label the action as COAGULATE_HEMOSTASIS.

8. Clipping.
- Sometimes a clipper may be trying to place a clip on the cystic duct or the cystic artery.
- In that case, label the action as CLIP and choose the appropriate target.

9. Sweeping.
- If the tool only sweeps across tissue, without actually dissecting or cutting, then the action may be sweeping.
- This can include sweeping blood or lightly moving across tissue without true dissection.

10. Retracting.
- Sometimes the tool is retracting rather than dissecting.
- This means it is gently pushing or holding tissue to improve visualization, without actually cutting or dissecting it.\n'''.strip()
    return prompt + '\n\n' + left_rule + '\n\n' + right_rule

prompts = {method: f'per-record CVS-context v3 debug wrapper around {method.removesuffix("_cvs_context")}' for method in METHODS}
print('records:', len(records))
print('methods:', METHODS)
print('left options:', simple_options['left_retraction_direction_code_options'])
right_triplets = simple_options['right_triplet_options']
print('right tool_type options:', sorted({row[0] for row in right_triplets}))
print('right action_code options:', sorted({row[1] for row in right_triplets}))
print('right target_structure options:', sorted({row[2] for row in right_triplets}))
print('right target contexts:', simple_options['interface_options']['target_context'])
print('camera options:', simple_options['camera_action_code_options'])
print('other options:', simple_options['other_action_code_options'])


records: 1312
methods: ['structured_prediction_cvs_context']
left options: ['KEEP_RETRACT_LATERAL', 'KEEP_RETRACT_MEDIAL', 'KEEP_RETRACT_UPWARD', 'RETRACT_LATERAL', 'RETRACT_MEDIAL', 'RETRACT_UPWARD', 'RETRACT_LATERAL_TO_MEDIAL', 'RETRACT_LATERAL_TO_UPWARD', 'RETRACT_MEDIAL_TO_LATERAL', 'RETRACT_MEDIAL_TO_UPWARD', 'RETRACT_UPWARD_TO_LATERAL', 'RETRACT_UPWARD_TO_MEDIAL']
right tool_type options: ['Hook', 'Irrigator', 'Maryland', 'Scissors', 'clipper']
right action_code options: ['CLIP', 'COAGULATE_HEMOSTASIS', 'COUNTERTRACTION_ASSIST', 'DISSECT', 'IRRIGATOR_ASPIRATE', 'RETRACT_DOWNWARD', 'SWEEPING', 'TOOL_WITHDRAW_UNBLOCKS_VIEW']
right target_structure options: ['CysticArtery', 'CysticDuct', 'CysticPlate', 'GallbladderNeck_Infundibulum', 'HepatocysticTriangle']
right target contexts: ['between presumed cystic duct and presumed cystic artery', 'between cystic artery and cystic plate', 'near the base of the hepatocystic triangle', 'close to the gallbladder neck', 'near the cystic duct', '

In [ ]:

def cache_get(path, default=None):
    return read_json(path, default if default is not None else {})


def cache_set(path, obj):
    write_json(path, obj)


def model_json_call(messages, cache_path, cache_key, signature, *, force=False, repair_label=None):
    cache = cache_get(cache_path, {})
    entry = cache.get(cache_key)
    if force or not entry or entry.get('signature') != signature:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=REQUEST_TEMPERATURE,
            extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
        )
        answer_raw = response.choices[0].message.content
        try:
            answer_json = extract_json_object(answer_raw)
            parse_error = None
        except Exception as exc:
            parse_error = repr(exc)
            malformed_path = ARTIFACT_DIR / f'malformed_{repair_label or "json"}_{cache_key.replace("/", "_").replace("::", "__")}.txt'
            malformed_path.write_text(answer_raw)
            repair_messages = [
                {'role': 'system', 'content': 'Repair malformed JSON. Return valid JSON only.'},
                {'role': 'user', 'content': f'Parse error: {parse_error}\n\nMalformed output:\n{answer_raw}'},
            ]
            repair_response = client.chat.completions.create(
                model=MODEL_ID,
                messages=repair_messages,
                temperature=0,
                extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
            )
            repaired_raw = repair_response.choices[0].message.content
            (ARTIFACT_DIR / f'repaired_{repair_label or "json"}_{cache_key.replace("/", "_").replace("::", "__")}.txt').write_text(repaired_raw)
            answer_json = extract_json_object(repaired_raw)
        entry = {'answer_raw': answer_raw, 'answer_json': answer_json, 'signature': signature, 'parse_error_repaired': parse_error}
        cache[cache_key] = entry
        cache_set(cache_path, cache)
    return entry


def run_segmentation(record, method_name):
    messages, prompt_record = build_messages_for_record(
        record,
        prompt_for_record(record, method_name),
        FRAMES_DIR,
        MODEL_ID,
        REQUEST_TEMPERATURE,
        REQUEST_ENABLE_THINKING,
        frame_labels=build_frame_labels_for_record(record),
    )
    prompt_method = segmentation_source_method(method_name)
    prompt_record['method'] = prompt_method
    signature = hashlib.sha256(json.dumps(prompt_record, sort_keys=True).encode('utf-8')).hexdigest()
    cache_key = f'{record["example_id"]}::{prompt_method}'
    cache = cache_get(SEGMENT_CACHE_PATH, {})
    entry = cache.get(cache_key)
    if FORCE_SEGMENT or not entry or entry.get('signature') != signature:
        response = client.chat.completions.create(
            model=MODEL_ID,
            messages=messages,
            temperature=REQUEST_TEMPERATURE,
            extra_body={'chat_template_kwargs': {'enable_thinking': REQUEST_ENABLE_THINKING}},
        )
        entry = {'answer_raw': response.choices[0].message.content, 'prompt_record': prompt_record, 'signature': signature}
        cache[cache_key] = entry
        cache_set(SEGMENT_CACHE_PATH, cache)
    return entry


def run_extraction(record, method_name, segmentation_entry):
    frame_range = [int(record['coarse']['start_frame']), int(record['coarse']['end_frame'])]
    cache_key = f'{record["example_id"]}::{method_name}'
    if is_deterministic_structured_method(method_name):
        answer_json = build_deterministic_structured_extraction(method_name, segmentation_entry['answer_raw'], frame_range[0], frame_range[1])
        signature = hashlib.sha256(json.dumps({'run_id': 'structured_prediction_deterministic_extraction_v1_cvsctx_v3_debug5', 'segmentation_signature': segmentation_entry.get('signature'), 'answer_json': answer_json}, sort_keys=True).encode('utf-8')).hexdigest()
        cache = cache_get(EXTRACTION_CACHE_PATH, {})
        entry = cache.get(cache_key)
        if FORCE_EXTRACTION or not entry or entry.get('signature') != signature:
            entry = {'answer_raw': json.dumps(answer_json, indent=2), 'answer_json': answer_json, 'signature': signature}
    else:
        prompt = build_extraction_prompt(record['example_id'], frame_range, method_name, segmentation_entry['answer_raw'], simple_options)
        signature = hashlib.sha256(json.dumps({'run_id': 'taxonomy_extraction_v1_cvsctx_v3_debug5', 'model': MODEL_ID, 'prompt': prompt}, sort_keys=True).encode('utf-8')).hexdigest()
        entry = model_json_call(
            [
                {'role': 'system', 'content': 'Extract constrained timestamped surgical actions. Return valid JSON only.'},
                {'role': 'user', 'content': prompt},
            ],
            EXTRACTION_CACHE_PATH,
            cache_key,
            signature,
            force=FORCE_EXTRACTION,
            repair_label='extraction',
        )
    entry['normalized'] = normalize_extraction(entry['answer_json'], frame_range[0], frame_range[1], simple_options)
    entry['normalized_natural_language'] = naturalize_simple_actions(entry['normalized'])
    cache = cache_get(EXTRACTION_CACHE_PATH, {})
    cache[cache_key] = entry
    cache_set(EXTRACTION_CACHE_PATH, cache)
    return entry


In [ ]:

results_payload = cache_get(RESULTS_PATH, {'results': []})
results_by_key = {(r.get('example_id'), r.get('method')): r for r in results_payload.get('results', [])}
start_time = time.time()
total = len(records) * len(METHODS)
step = 0

progress = tqdm(total=total, desc='v3 segmentation/extraction', unit='run')
for record in records:
    for method_name in METHODS:
        step += 1
        progress.update(1)
        progress.set_postfix(method=method_name, example=record['example_id'][:8])
        key = (record['example_id'], method_name)
        existing = results_by_key.get(key)
        if existing and existing.get('status') == 'ok' and existing.get('labeling_version') == LABELING_VERSION and not (FORCE_SEGMENT or FORCE_EXTRACTION or FORCE_JUDGE):
            if step % 10 == 0 or step == total:
                print(f'[{step}/{total}] cached {method_name} {record["example_id"]}')
            continue
        try:
            seg_entry = run_segmentation(record, method_name)
            extraction_entry = run_extraction(record, method_name, seg_entry)
            item = {
                'status': 'ok',
                'labeling_version': LABELING_VERSION,
                'example_id': record['example_id'],
                'video_id': record['video_id'],
                'criterion': record.get('criterion'),
                'method': method_name,
                'segmentation_answer': seg_entry['answer_raw'],
                'extracted_actions': extraction_entry['normalized'],
                'extracted_actions_natural_language': extraction_entry['normalized_natural_language'],
            }
        except Exception as exc:
            item = {
                'status': 'error',
                'labeling_version': LABELING_VERSION,
                'example_id': record.get('example_id'),
                'video_id': record.get('video_id'),
                'criterion': record.get('criterion'),
                'method': method_name,
                'error': repr(exc),
            }
            print('ERROR', item['example_id'], method_name, repr(exc))
        results_by_key[key] = item
        cache_set(RESULTS_PATH, {'results': list(results_by_key.values())})
        print(f'[{step}/{total}] {item["status"]} {method_name} {record["example_id"]}')

progress.close()
elapsed = (time.time() - start_time) / 60
if WRITE_DERIVED_ARTIFACTS:
    cache_set(RESULTS_PATH, {'results': list(results_by_key.values())})
print(f'done/paused after {elapsed:.1f} min')
print('ok:', sum(1 for r in results_by_key.values() if r.get('status') == 'ok'), 'errors:', sum(1 for r in results_by_key.values() if r.get('status') == 'error'))


In [7]:
1


1

---

# Apply v3.1 override and export final synthetic labels


# Qwen3.5 Audit v11 CVS Context v3.1 Train Full - v3.1 override/export

This section reads the split-specific base `cvsctx_v3` results, applies the v3.1 clip-rubric Hook-to-Maryland override where rubric-cache entries exist, and writes the final synthetic action JSON files.

Train uses generated SAGES train transition clip records at `train/audit_v11_clips`. Generate them with `scripts/generate_train_transition_clip_records.py --split train`.


In [ ]:
import json
import sys
from copy import deepcopy
from pathlib import Path

import pandas as pd
from tqdm.auto import tqdm
from IPython.display import display

ROOT_DIR = Path('/mnt/md0/weiqiuy/surgent')
SRC_DIR = ROOT_DIR / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from cvs_act.action_segment_eval import (
    convert_record_to_simple_actions,
    load_audit_records,
    naturalize_simple_actions,
    read_json,
    write_json,
)

ANNOTATION_ROOT = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_annotations/v1'
AUDIT_V11_DIR = ANNOTATION_ROOT / 'train/audit_v11_clips'
SOURCE_ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_train_full'
RUBRIC_CACHE_PATH = ROOT_DIR / 'notebooks/qwen3.5_audit_v11_seg_extract_spec_eval/artifacts/qwen3.5_audit_v11_right_tool_rubric_clip_eval_1fps_cache.json'
ARTIFACT_DIR = ROOT_DIR / 'notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_1_train_full'
SYNTHETIC_DATA_DIR = ROOT_DIR / 'data/processed/CVS_Challenge_SAGES_v1/cvs_act_synthetic_data/audit_v11_cvsctx_v3_1/train_full'
RESULTS_PATH = ARTIFACT_DIR / 'results.json'
SUMMARY_PATH = ARTIFACT_DIR / 'override_summary.json'
SYNTHETIC_GT_DEFAULT_METHOD = 'structured_prediction_cvs_context'
RUBRIC_METHOD = 'prompt2_all_in_one_json_with_shapes'
LABELING_VERSION = 'code_labels_v3_1_cvsctx_from_v3_train_full_hook_to_maryland'
WRITE_DERIVED_ARTIFACTS = True

ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)
SYNTHETIC_DATA_DIR.mkdir(parents=True, exist_ok=True)
print('source:', SOURCE_ARTIFACT_DIR)
print('rubric cache:', RUBRIC_CACHE_PATH)
print('annotations:', AUDIT_V11_DIR)
print('artifacts:', ARTIFACT_DIR)
print('synthetic data:', SYNTHETIC_DATA_DIR)


In [9]:
records = load_audit_records(AUDIT_V11_DIR)
gt_simple_records = [convert_record_to_simple_actions(record) for record in records]
source_results = read_json(SOURCE_ARTIFACT_DIR / 'results.json', {'results': []})['results']
rubric_cache = read_json(RUBRIC_CACHE_PATH, {})


def normalize_tool_label(value):
    if value is None:
        return ''
    value = str(value).strip()
    if not value:
        return ''
    mapping = {
        'grasper': 'Grasper',
        'maryland': 'Maryland',
        'hook': 'Hook',
        'irrigator': 'Irrigator',
        'scissors': 'Scissors',
        'clipper': 'Clipper',
        'unknown': 'Unknown',
        '(absent)': '(absent)',
        '(not set)': '(absent)',
        'not set': '(absent)',
        'none': '(absent)',
    }
    return mapping.get(value.lower(), value)


def clip_tool_from_segments(segments):
    tools = sorted({
        normalize_tool_label(seg.get('tool_type'))
        for seg in (segments or [])
        if normalize_tool_label(seg.get('tool_type'))
    })
    if not tools:
        return '(absent)', '(absent)'
    if len(tools) == 1:
        return tools[0], tools[0]
    return '(multi)', ' | '.join(tools)


def get_rubric_tool(example_id):
    payload = rubric_cache.get(example_id, {}).get(RUBRIC_METHOD, {})
    return normalize_tool_label((payload.get('aggregation') or {}).get('predicted_tool', ''))


def rewrite_right_hook_to_maryland(extracted_actions):
    updated = deepcopy(extracted_actions)
    changed = False
    for seg in updated.get('right', []) or []:
        if normalize_tool_label(seg.get('tool_type')) != 'Hook':
            continue
        seg['tool_type'] = 'Maryland'
        triplet = list(seg.get('triplet') or [])
        if triplet:
            triplet[0] = 'Maryland'
            seg['triplet'] = triplet
        changed = True
    return updated, changed


methods = sorted({row.get('method') for row in source_results if row.get('status') == 'ok'})
print('source rows:', len(source_results))
print('methods:', methods)


missing_rubric_examples = [
    row.get('example_id')
    for row in source_results
    if row.get('status') == 'ok' and row.get('example_id') not in rubric_cache
]
if missing_rubric_examples:
    print(
        'warning: rubric cache has no entries for',
        len(set(missing_rubric_examples)),
        'example(s); v3.1 override will not apply to those examples.'
    )


source rows: 1314
methods: ['description_only_cvs_context', 'structured_prediction_cvs_context']


In [10]:
derived_results = []
override_rows = []

for row in tqdm(source_results, desc='v3.1 override', unit='row'):
    item = deepcopy(row)
    if item.get('status') != 'ok':
        derived_results.append(item)
        continue

    clip_tool_name, clip_tool_set_text = clip_tool_from_segments(item.get('extracted_actions', {}).get('right', []))
    rubric_tool_name = get_rubric_tool(item['example_id'])
    applied = False

    if clip_tool_name == 'Hook' and rubric_tool_name == 'Maryland':
        updated_actions, changed = rewrite_right_hook_to_maryland(item['extracted_actions'])
        if changed:
            item['extracted_actions'] = updated_actions
            item['extracted_actions_natural_language'] = naturalize_simple_actions(updated_actions)
            applied = True

    item['labeling_version'] = LABELING_VERSION
    item['v3_1_override'] = {
        'source_result_dir': str(SOURCE_ARTIFACT_DIR),
        'rule': 'if clip-level v3 right tool is Hook and best rubric clip tool is Maryland, rewrite right Hook tool labels to Maryland',
        'source_clip_right_tool': clip_tool_name,
        'source_clip_right_tool_set_text': clip_tool_set_text,
        'rubric_method': RUBRIC_METHOD,
        'rubric_clip_tool': rubric_tool_name,
        'applied': applied,
    }
    derived_results.append(item)

    override_rows.append({
        'example_id': item.get('example_id'),
        'video_id': item.get('video_id'),
        'criterion': item.get('criterion'),
        'method': item.get('method'),
        'source_clip_right_tool': clip_tool_name,
        'rubric_clip_tool': rubric_tool_name,
        'override_applied': applied,
    })

if WRITE_DERIVED_ARTIFACTS:
    write_json(RESULTS_PATH, {'results': derived_results})
    write_json(SUMMARY_PATH, override_rows)

    synthetic_gt_by_method = {}
    for item in tqdm(derived_results, desc='v3.1 synthetic export', unit='row'):
        if item.get('status') != 'ok':
            continue
        method_name = item.get('method')
        if not method_name:
            continue
        pred = dict(item.get('extracted_actions', {}))
        pred['example_id'] = item['example_id']
        pred['video_id'] = item['video_id']
        pred['criterion'] = item.get('criterion')
        synthetic_gt_by_method.setdefault(method_name, []).append(pred)

    for method_name, records in synthetic_gt_by_method.items():
        method_slug = method_name.replace('/', '__')
        write_json(SYNTHETIC_DATA_DIR / f'synthetic_audit_v11_simple_actions__{method_slug}.json', records)

    default_records = synthetic_gt_by_method.get(SYNTHETIC_GT_DEFAULT_METHOD)
    if default_records:
        write_json(SYNTHETIC_DATA_DIR / 'synthetic_audit_v11_simple_actions.json', default_records)

summary_df = pd.DataFrame(override_rows)
display(summary_df.groupby(['method', 'source_clip_right_tool', 'rubric_clip_tool'], dropna=False)['override_applied'].agg(['count', 'sum']).reset_index().sort_values(['method', 'sum', 'count'], ascending=[True, False, False]))
print('saved results:', RESULTS_PATH)


v3.1 override:   0%|          | 0/1314 [00:00<?, ?row/s]

v3.1 synthetic export:   0%|          | 0/1314 [00:00<?, ?row/s]

,method,source_clip_right_tool,rubric_clip_tool,count,sum
0,description_only_cvs_context,Hook,,2,0
4,structured_prediction_cvs_context,Hook,,630,0
2,structured_prediction_cvs_context,(multi),,427,0
6,structured_prediction_cvs_context,Maryland,,212,0
5,structured_prediction_cvs_context,Irrigator,,32,0
1,structured_prediction_cvs_context,(absent),,7,0
7,structured_prediction_cvs_context,Scissors,,3,0
3,structured_prediction_cvs_context,Clipper,,1,0


saved results: /mnt/md0/weiqiuy/surgent/notebooks/artifacts/qwen3.5_audit_v11_seg_extract_spec_eval_cvsctx_v3_1_train_full/results.json
